# Phase 1 — An image is just a grid of numbers

Before we touch any neural network, let's build the single most important intuition:

1. To a computer, an image is a **tensor** (a multi-dimensional array) of numbers.
2. A **convolution** slides a tiny grid of weights (a *filter*) over the image to detect a pattern.
3. A **CNN** is just a stack of these filters — except the network *learns* the filter values instead of us writing them by hand.

By the end of this notebook you'll have applied a hand-made edge detector to a real image and *seen* what a convolution produces. That's the whole idea behind every CNN.

In [ ]:
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

print("torch version:", torch.__version__)

## 1. Grab one image

We'll download a single sample image that ships with torchvision so we don't need the dataset yet.

In [ ]:
from torchvision.io import read_image
from torchvision.utils import draw_bounding_boxes  # noqa: F401 (just to verify torchvision works)
import urllib.request, os

os.makedirs("../data", exist_ok=True)
url = "https://raw.githubusercontent.com/pytorch/vision/main/gallery/assets/dog1.jpg"
path = "../data/sample.jpg"
if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)

img = read_image(path)  # shape: [channels, height, width], values 0-255
print("This image is a tensor with shape:", tuple(img.shape))
print("That's [color_channels, height, width] -> 3 channels (Red, Green, Blue)")
print("Data type:", img.dtype, "| min:", img.min().item(), "max:", img.max().item())

## 2. Look at the actual numbers

Let's print a tiny 8x8 corner of the Red channel. **This is what the model sees** — no pixels, just numbers from 0 (dark) to 255 (bright).

In [ ]:
print("Top-left 8x8 corner of the RED channel:\n")
print(img[0, :8, :8])

In [ ]:
# And here's the image those numbers represent.
# matplotlib wants [height, width, channels], so we move the channel axis to the end.
plt.imshow(img.permute(1, 2, 0))
plt.title("The original image")
plt.axis("off")
plt.show()

## 3. Apply a convolution by hand

A convolution filter is a small grid of numbers. We slide it across the image; at each position we multiply overlapping numbers and sum them up. Different filters detect different patterns.

Below is a classic **vertical edge detector**. Where the image changes from dark to bright horizontally, the output lights up. Watch what it does.

In [ ]:
# Convert to grayscale and to float so the math is clean.
gray = img.float().mean(dim=0, keepdim=True) / 255.0  # shape [1, H, W]
gray = gray.unsqueeze(0)  # add a batch dimension -> [1, 1, H, W]

# A 3x3 vertical-edge filter (this is a Sobel filter).
edge_filter = torch.tensor([
    [-1.0, 0.0, 1.0],
    [-2.0, 0.0, 2.0],
    [-1.0, 0.0, 1.0],
]).view(1, 1, 3, 3)

# This single line IS a convolution. A CNN does this thousands of times.
edges = F.conv2d(gray, edge_filter, padding=1)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(gray.squeeze(), cmap="gray");  ax[0].set_title("Input (grayscale)");  ax[0].axis("off")
ax[1].imshow(edges.squeeze().abs(), cmap="gray");  ax[1].set_title("After the edge filter");  ax[1].axis("off")
plt.show()

## 4. Your turn — experiment

Try these and re-run the cell above:

- **Rotate the filter** to detect *horizontal* edges instead (transpose the numbers).
- **A blur filter**: set every value to `1/9` in a 3x3 grid. What happens?
- **Make the filter bigger** (5x5) and see how the output changes.

### The key takeaway

We *chose* those filter numbers to detect edges. In a real CNN, the numbers start out **random**, and training nudges them — via the loop you'll write in Phase 3 — until they detect whatever is useful for telling a hotdog from not-a-hotdog: edges, then textures, then shapes, then "sausage-in-a-bun."

Next up: **Phase 2**, where we load the real hotdog dataset. Run `02_data.ipynb`.